<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.6-rag-with-llamaindex/notebooks/exercises-6.6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.6 — RAG with LlamaIndex
Netsetos GenAI Course v2.0 | Module 6

LlamaParse, IngestionPipeline, response synthesis, query engines, Workflows, agents, production


In [ ]:
# pip install llama-index llama-index-llms-openai llama-index-vector-stores-chroma llama-parse -q
print('Ready for LlamaIndex RAG')


## Ex 1: LlamaIndex RAG in 5 Lines


In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model='gpt-4o-mini')

docs = [Document(text='RAG grounds LLM responses in external documents.'),
        Document(text='LlamaIndex uses Documents, Nodes, and Indexes.'),
        Document(text='IngestionPipeline handles caching and dedup automatically.')]

index = VectorStoreIndex.from_documents(docs)
engine = index.as_query_engine(response_mode='compact')
response = engine.query('What is LlamaIndex?')
print(response)


## Ex 2: Response Mode Comparison


In [ ]:
for mode in ['compact', 'refine', 'tree_summarize']:
    engine = index.as_query_engine(response_mode=mode)
    response = engine.query('Summarize all documents')
    print(f'\n=== {mode} ===')
    print(f'Response: {str(response)[:100]}...')
    # In production: track LLM calls and latency per mode

print('\nDecision:')
print('  compact (default): 1-2 LLM calls, fastest, general QA')
print('  tree_summarize: log(N) calls, no sequential bias, summarization')
print('  refine: N calls, slowest, every-chunk-matters analysis')


## Ex 3: IngestionPipeline with Caching


In [ ]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.extractors import TitleExtractor
from llama_index.embeddings.openai import OpenAIEmbedding

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=512, chunk_overlap=20),
        TitleExtractor(),
        OpenAIEmbedding(),
    ],
)

# First run - processes all documents
nodes = pipeline.run(documents=docs)
print(f'First run: {len(nodes)} nodes processed')

# Second run - skips unchanged (caching)
nodes2 = pipeline.run(documents=docs)
print(f'Second run: {len(nodes2)} nodes (cached, near-instant)')

# Persist cache
pipeline.persist('./pipeline_cache')
print('Cache persisted to ./pipeline_cache')


## Ex 4: RouterQueryEngine


In [ ]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import PydanticSingleSelector
from llama_index.core.tools import QueryEngineTool
from llama_index.core import SummaryIndex

# Two different indexes
summary_index = SummaryIndex.from_documents(docs)
vector_index = VectorStoreIndex.from_documents(docs)

# Router dispatches based on query type
# router = RouterQueryEngine(
#     selector=PydanticSingleSelector.from_defaults(),
#     query_engine_tools=[
#         QueryEngineTool.from_defaults(
#             summary_index.as_query_engine(),
#             description='For summarization questions about all documents'),
#         QueryEngineTool.from_defaults(
#             vector_index.as_query_engine(),
#             description='For specific fact retrieval from documents'),
#     ],
# )

print('RouterQueryEngine:')
print('  "Summarize all docs" → routes to SummaryIndex')
print('  "What is RAG?" → routes to VectorStoreIndex')
print('  Tool descriptions drive routing quality')


## Ex 5: SubQuestionQueryEngine


In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata

# Two separate indexes for different data sources
# sub_engine = SubQuestionQueryEngine.from_defaults(
#     query_engine_tools=[
#         QueryEngineTool(company_a_engine, ToolMetadata(
#             name='company_a', description='Financial data for Company A')),
#         QueryEngineTool(company_b_engine, ToolMetadata(
#             name='company_b', description='Financial data for Company B')),
#     ],
#     use_async=True,  # Parallel sub-question execution
# )

print('SubQuestionQueryEngine:')
print('  Input: "Compare revenue of Company A vs B"')
print('  Generated sub-questions:')
print('    1. "What is Company A Q3 revenue?" → company_a tool')
print('    2. "What is Company B Q3 revenue?" → company_b tool')
print('  Synthesized: comparative answer from both results')


## Ex 6: FunctionAgent with Tools


In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool, QueryEngineTool

def calculate(expression: str) -> str:
    '''Calculate a math expression. Returns the result.'''
    try: return str(eval(expression))
    except: return 'Error evaluating expression'

calc_tool = FunctionTool.from_defaults(calculate)
query_tool = QueryEngineTool.from_defaults(
    index.as_query_engine(),
    name='search_docs',
    description='Search internal documentation'
)

# agent = FunctionAgent(
#     tools=[calc_tool, query_tool],
#     llm=OpenAI(model='gpt-4o'),
#     system_prompt='You are a helpful research assistant.'
# )
# response = await agent.run(user_msg='What is RAG?')

print('FunctionAgent: preferred for production (native tool calling)')
print('ReActAgent: works with any LLM (Thought→Action→Observation)')
print('AgentWorkflow: multi-agent with can_handoff_to')


## Ex 7: Evaluation Pipeline


In [ ]:
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

# evaluators
faith_eval = FaithfulnessEvaluator()
rel_eval = RelevancyEvaluator()

# Evaluate a response
engine = index.as_query_engine()
response = engine.query('What is LlamaIndex?')

# faith = await faith_eval.aevaluate_response(response=response)
# rel = await rel_eval.aevaluate_response(query='What is LlamaIndex?', response=response)

print('Evaluation:')
print('  FaithfulnessEvaluator: is answer grounded in context?')
print('  RelevancyEvaluator: does context address the query?')
print('  CorrectnessEvaluator: score vs reference (1-5, needs labels)')
print('  BatchEvalRunner: async batch across 8+ workers')
print('  generate_question_context_pairs(): synthetic test data')


## Ex 8: Storage & Settings


In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

# Persist
index.storage_context.persist('./storage')
print('Index persisted to ./storage')

# Load
# storage_ctx = StorageContext.from_defaults(persist_dir='./storage')
# loaded_index = load_index_from_storage(storage_ctx)

# Global Settings (replaces ServiceContext)
from llama_index.core import Settings
# Settings.llm = OpenAI(model='gpt-4o')
# Settings.embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-m3')  # Hindi
# Settings.chunk_size = 512

print('\nSettings replaces deprecated ServiceContext')
print('Set once globally, applies to all LlamaIndex operations')
print('\nProduction vector stores: Qdrant, Pinecone, Weaviate')
print('  Auto-persistent — persist() is a no-op')


---
## Recap
LlamaIndex: document-centric RAG framework. Documents → Nodes → Index → Query Engine. LlamaParse: best-in-class PDF parsing (4 tiers, 130+ formats, ChrF++ 81%). IngestionPipeline: declarative processing with per-node hash caching and DocStore dedup. Response modes: compact (default, QA), tree_summarize (summarization), refine (every-chunk-matters). Advanced query engines: RouterQueryEngine (dispatch), SubQuestionQueryEngine (decomposition), CitationQueryEngine (source attribution). Workflows 1.0: event-driven @step orchestration (vs LangGraph's state machines). Agents: FunctionAgent (preferred, native tool calling) + AgentWorkflow (multi-agent handoff). Production: external vector store, Redis-backed IngestionPipeline, OpenTelemetry with Langfuse, FaithfulnessEvaluator + RelevancyEvaluator. Hybrid: LlamaIndex for data + LangChain for orchestration.
